In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 1 – SETUP E IMPORTS                                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import List, Dict
from extract.sheets_fetcher import SheetsFetcher
import functools
from IPython.display import display  # para mostrar DataFrames no notebook

# ─── Configurações globais ────────────────────────────────────────────────
PLANILHA_ID: str = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
CREDS_PATH: Path = Path("creds.json")
MODEL_TABS: List[str] = [
    "modeloGeral",
    "modeloGenero",
    "modeloRegiao",
    "modeloAlcance",
    "modeloIdade",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(asctime)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ─── Exceções customizadas ────────────────────────────────────────────────
class MismatchingImpressionsError(Exception):
    """Raised when impression totals differ across sheets."""

class PostMergeImpressionError(Exception):
    """Raised when merged impression total does not match expectation."""


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 2 – LEITURA DAS ABAS-MODELO + NORMALIZAÇÃO                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Responsabilidades:
#  • Instanciar SheetsFetcher
#  • Carregar as cinco abas-modelo em dfs_raw
#  • Normalizar colunas:
#      - header → lowercase, strip
#      - renomear 'impressoes' → 'impressions'
#      - 'date' → datetime, 'impressions' → nullable Int64
#      - 'campanha' → strip(), casefold()
#      - 'veiculo' → strip(), casefold(), unificar espaços e remover espaços internos de plataformas conhecidas
#  • Logar shape e mostrar head para inspeção

import pandas as pd
import numpy as np
import logging
from IPython.display import display

# Instância SheetsFetcher já importada na Célula 1
logging.info("⚙️  Instanciando SheetsFetcher…")
fetcher = SheetsFetcher(
    spreadsheet_id=PLANILHA_ID,
    creds_path=str(CREDS_PATH),
    header_row=0,
    col_range="A:ZZ",
    cache_ttl=300,
)

logging.info("📥 Lendo abas-modelo: %s", ", ".join(MODEL_TABS))
dfs_raw = fetcher.get(MODEL_TABS)

# Normalização e coerção
dfs: dict[str, pd.DataFrame] = {}
for tab, df in dfs_raw.items():
    # 1) Colunas lowercase e strip
    df.columns = [c.strip().lower() for c in df.columns]
    # 2) Rename impressoes → impressions
    if "impressoes" in df.columns:
        df.rename(columns={"impressoes": "impressions"}, inplace=True)
    # 3) Coerce date
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    # 4) Coerce impressions to numeric nullable
    if "impressions" in df.columns:
        s = pd.to_numeric(df["impressions"], errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan)
        df["impressions"] = s.fillna(0).astype("Int64")
    # 5) Normalize campanha
    if "campanha" in df.columns:
        df["campanha"] = (
            df["campanha"].astype(str)
            .str.strip()
            .str.casefold()
            .str.replace(r"\s+", " ", regex=True)
        )
    # 6) Normalize veiculo
    if "veiculo" in df.columns:
        df["veiculo"] = (
            df["veiculo"].astype(str)
            .str.strip()
            .str.casefold()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace("tik tok", "tiktok", regex=False)
        )
    dfs[tab] = df
    logging.info("✅ Aba '%s' normalizada – %d linhas × %d colunas", tab, *df.shape)
    display(df.head(3))

# dfs agora contém todos os DataFrames prontos para validações, filtros e merges.


INFO | 2025-06-17 16:05:11 | ⚙️  Instanciando SheetsFetcher…
INFO | 2025-06-17 16:05:11 | 📥 Lendo abas-modelo: modeloGeral, modeloGenero, modeloRegiao, modeloAlcance, modeloIdade
INFO | 2025-06-17 16:05:11 | 🔄 batchGet tentativa para ranges: ['modeloGeral!A:ZZ', 'modeloGenero!A:ZZ', 'modeloRegiao!A:ZZ', 'modeloAlcance!A:ZZ', 'modeloIdade!A:ZZ']
INFO | 2025-06-17 16:05:15 | 🔍 range completo retornado pela API: modeloGeral!A1:Z2320
INFO | 2025-06-17 16:05:15 | 🔍 range completo retornado pela API: modeloGenero!A1:O2362
INFO | 2025-06-17 16:05:15 | 🔍 range completo retornado pela API: modeloRegiao!A1:O34252
INFO | 2025-06-17 16:05:15 | 🔍 range completo retornado pela API: modeloAlcance!A1:L1845
INFO | 2025-06-17 16:05:15 | 🔍 range completo retornado pela API: modeloIdade!A1:O8403
INFO | 2025-06-17 16:05:15 | 📡 batchGet 5 ranges
INFO | 2025-06-17 16:05:15 | ✅ Aba 'modeloGeral' normalizada – 1887 linhas × 26 colunas


,date,account_name,campanha,id_campanha,veiculo,id_veiculo,ad_group_name,ad_name,start,end,...,video_play,video_watched_25,video_watched_50,video_watched_75,video_watched_100,post_reactions,post_shares,post_comments,engajamento_total,id
0,2025-03-01,Sebrae Nacional - DEBRITO,catalisa ict,dbt_sbrae_2025_catalisa,instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,10091,7641,4014,291,24,3,21,0,24,"2025-03-01-Catalisa ICT-144387-1914,79-3349"
1,2025-03-01,Sebrae Nacional - DEBRITO,catalisa ict,dbt_sbrae_2025_catalisa,instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,12370,9856,1339,148,118,41,31,0,72,"2025-03-01-Catalisa ICT-173584-1950,14-3344"
2,2025-03-02,Sebrae Nacional - DEBRITO,catalisa ict,dbt_sbrae_2025_catalisa,instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,8521,6380,3159,354,30,3,14,0,17,"2025-03-02-Catalisa ICT-161093-1898,22-3143"


INFO | 2025-06-17 16:05:15 | ✅ Aba 'modeloGenero' normalizada – 2361 linhas × 15 colunas


,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,gender,impressions,cost,link_clicks,video_watched_100,id
0,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,Mulher,89078,"1103,91",1916,12,"2025-03-01-Catalisa ICT-89078-1103,91-1916"
1,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,Tráfego,Mulher,118553,"1276,86",2216,96,"2025-03-01-Catalisa ICT-118553-1276,86-2216"
2,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,Homem,55043,"806,63",1427,12,"2025-03-01-Catalisa ICT-55043-806,63-1427"


INFO | 2025-06-17 16:05:15 | ✅ Aba 'modeloRegiao' normalizada – 34251 linhas × 15 colunas


,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,region,impressions,cost,link_clicks,video_watched_100,id
0,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_CARD_KV_ACAO_SBRAE_2025_PSMN0293,Tráfego,Acre,532,"1,71",4,0,"2025-06-15-PSMN-532-1,71-4"
1,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,Tráfego,Acre,26,"0,18",1,0,"2025-06-15-PSMN-26-0,18-1"
2,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,Tráfego,Acre,172,"1,04",6,0,"2025-06-15-PSMN-172-1,04-6"


INFO | 2025-06-17 16:05:15 | ✅ Aba 'modeloAlcance' normalizada – 1844 linhas × 12 colunas


,date,account_name,veiculo,id_veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,reach,impressions,id
0,2025-03-01,Sebrae Nacional - DEBRITO,instagram,2,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,124036,144387,2025-03-01-Catalisa ICT-144387--
1,2025-03-01,Sebrae Nacional - DEBRITO,instagram,2,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,Tráfego,140005,173584,2025-03-01-Catalisa ICT-173584--
2,2025-03-02,Sebrae Nacional - DEBRITO,instagram,2,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,128226,161093,2025-03-02-Catalisa ICT-161093--


INFO | 2025-06-17 16:05:15 | ✅ Aba 'modeloIdade' normalizada – 8402 linhas × 15 colunas


,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,age,impressions,cost,link_clicks,video_watched_100,id
0,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,25-34,25356,"157,48",233,3,"2025-03-01-Catalisa ICT-25356-157,48-233"
1,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,Tráfego,25-34,29151,"148,19",198,10,"2025-03-01-Catalisa ICT-29151-148,19-198"
2,2025-03-01,Sebrae Nacional - DEBRITO,2,instagram,dbt_sbrae_2025_catalisa,catalisa ict,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,Tráfego,35-44,61924,"622,43",1026,12,"2025-03-01-Catalisa ICT-61924-622,43-1026"


In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 3 – VALIDAÇÃO DE IMPRESSÕES POR PLATAFORMA (ORIGEM)              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import re
from typing import Dict, List
import pandas as pd
import numpy as np
import logging
from IPython.display import display

# ─── Garantir que `fetcher` exista ─────────────────────────────────────────
try:
    fetcher
except NameError:
    fetcher = SheetsFetcher(
        spreadsheet_id=PLANILHA_ID,
        creds_path=str(CREDS_PATH),
        header_row=0,
        col_range="A:ZZ",
        cache_ttl=300,
    )

# ─── Abas de ORIGEM ────────────────────────────────────────────────────────
ORIGIN_TABS: List[str] = [
    "pinterestGeral", "pinterestGenero", "pinterestRegiao",
    "pinterestAlcance", "pinterestIdade",
    "metaGeral", "metaAlcance", "metaIdade", "metaRegiao", "metaGenero",
    "tiktokGeral", "tiktokAlcance", "tiktokIdade", "tiktokRegiao", "tiktokGenero",
    "linkedinGeral", "linkedinAlcance", "linkedinRegiao",
]

def _coerce_impressions(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce 'date' to datetime and safely handle 'impressions' to Int64."""
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if "impressions" in df.columns:
        s = pd.to_numeric(df["impressions"], errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan)
        df["impressions"] = s.fillna(0).astype("Int64")
    return df

# ─── Carregar e coerir abas de ORIGEM ─────────────────────────────────────
logging.info("📥 Carregando abas de ORIGEM para validação")
origin_raw = fetcher.get(ORIGIN_TABS)
origin_dfs = {tab: _coerce_impressions(df) for tab, df in origin_raw.items()}

# ─── Validação por plataforma ───────────────────────────────────────────────
platforms = sorted({re.match(r"^([a-z]+)", tab).group(1) for tab in ORIGIN_TABS})

for platform in platforms:
    # Observação: em LinkedIn, a aba de região ignora localidades não determinadas,
    # por isso o total de 'linkedinRegiao' costuma ser menor que o geral.
    group = {tab: origin_dfs[tab] for tab in origin_dfs if tab.startswith(platform)}
    totals = {tab: int(df["impressions"].sum()) for tab, df in group.items()}

    summary_df = (
        pd.DataFrame.from_dict(totals, orient="index", columns=["impressions"])
          .reset_index()
          .rename(columns={"index": "Aba"})
    )
    mode_vals = summary_df["impressions"].mode()
    mode_val = mode_vals.iloc[0] if not mode_vals.empty else summary_df["impressions"].iloc[0]
    summary_df["divergence"] = summary_df["impressions"] - mode_val

    print(f"\n🔷 Validação ORIGEM – plataforma {platform.upper()}")
    display(summary_df)

    # Se houver divergência, apenas warning (não interrompe)
    divergent = summary_df.loc[summary_df["divergence"] != 0, "Aba"].tolist()
    if divergent:
        logging.warning(
            "⚠️ Divergência na ORIGEM [%s] em abas: %s",
            platform, ", ".join(divergent)
        )
    else:
        logging.info(
            "✅ Plataforma %s: todos os totais de impressions são iguais (%d)",
            platform, mode_val
        )


INFO | 2025-06-17 16:05:15 | 📥 Carregando abas de ORIGEM para validação
INFO | 2025-06-17 16:05:15 | 🔄 batchGet tentativa para ranges: ['pinterestGeral!A:ZZ', 'pinterestGenero!A:ZZ', 'pinterestRegiao!A:ZZ', 'pinterestAlcance!A:ZZ', 'pinterestIdade!A:ZZ', 'metaGeral!A:ZZ', 'metaAlcance!A:ZZ', 'metaIdade!A:ZZ', 'metaRegiao!A:ZZ', 'metaGenero!A:ZZ', 'tiktokGeral!A:ZZ', 'tiktokAlcance!A:ZZ', 'tiktokIdade!A:ZZ', 'tiktokRegiao!A:ZZ', 'tiktokGenero!A:ZZ', 'linkedinGeral!A:ZZ', 'linkedinAlcance!A:ZZ', 'linkedinRegiao!A:ZZ']
INFO | 2025-06-17 16:05:17 | 🔍 range completo retornado pela API: pinterestGeral!A1:U1047
INFO | 2025-06-17 16:05:17 | 🔍 range completo retornado pela API: pinterestGenero!A1:M284
INFO | 2025-06-17 16:05:17 | 🔍 range completo retornado pela API: pinterestRegiao!A1:M2582
INFO | 2025-06-17 16:05:17 | 🔍 range completo retornado pela API: pinterestAlcance!A1:K1047
INFO | 2025-06-17 16:05:17 | 🔍 range completo retornado pela API: pinterestIdade!A1:M1186
INFO | 2025-06-17 16:05:1


🔷 Validação ORIGEM – plataforma LINKEDIN


,Aba,impressions,divergence
0,linkedinGeral,5596048,0
1,linkedinAlcance,5596048,0
2,linkedinRegiao,3018092,-2577956


WARNING | 2025-06-17 16:05:17 | ⚠️ Divergência na ORIGEM [linkedin] em abas: linkedinRegiao



🔷 Validação ORIGEM – plataforma META


...

,Aba,impressions,divergence
0,pinterestGeral,23730100,0
1,pinterestGenero,23730100,0
2,pinterestRegiao,23730100,0
3,pinterestAlcance,23730100,0
4,pinterestIdade,23730100,0


INFO | 2025-06-17 16:05:17 | ✅ Plataforma pinterest: todos os totais de impressions são iguais (23730100)



🔷 Validação ORIGEM – plataforma TIKTOK


,Aba,impressions,divergence
0,tiktokGeral,99968436,4107
1,tiktokAlcance,99968436,4107
2,tiktokIdade,99964329,0
3,tiktokRegiao,92107264,-7857065
4,tiktokGenero,99964329,0


WARNING | 2025-06-17 16:05:17 | ⚠️ Divergência na ORIGEM [tiktok] em abas: tiktokGeral, tiktokAlcance, tiktokRegiao


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 4 – FILTRAGEM POR CAMPANHA + VEÍCULOS                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Responsabilidades:
#  • Filtrar cada DataFrame em `dfs` pela campanha e pelos veículos desejados
#  • Comparação case-insensitive de campanha e substring em veículo
#  • Se campaigns_filter vazio → mantém todas as campanhas
#  • Se vehicles_map_lc[campanha] vazio → mantém todos os veículos
#  • Incluir “meta” (Facebook+Instagram) como veículo permitido
#  • Exibir shape e top-3 linhas de cada DataFrame filtrado

import logging
from typing import List, Dict
from IPython.display import display

# ─── 1) Parâmetros de filtro ────────────────────────────────────────────────
campaigns_filter = ["2025_6_PSMN_TRAF_COMERCIALIZAÇÃO_CPC"]
vehicles_by_campaign = {
    "2025_6_PSMN_TRAF_COMERCIALIZAÇÃO_CPC": ["Meta", "TikTok", "Instagram", "Facebook"],
}

# ─── 2) Versões casefolded para matching ────────────────────────────────────
campaigns_filter_lc = [c.casefold() for c in campaigns_filter]
vehicles_map_lc = {
    camp.casefold(): [v.casefold() for v in lst]
    for camp, lst in vehicles_by_campaign.items()
}

# ─── 3) Função de filtragem ────────────────────────────────────────────────
def filter_df(df: pd.DataFrame, campaigns_lc: List[str], vehicles_map_lc: Dict[str, List[str]]) -> pd.DataFrame:
    # 1) Filtra por campanha (casefold)
    if campaigns_lc:
        df = df[df["campanha"].str.casefold().isin(campaigns_lc)].copy()

    # 2) Filtra por veículo (substring)
    def keep(row):
        allowed = vehicles_map_lc.get(row["campanha"].casefold(), [])
        if not allowed:
            return True
        return any(sub in row["veiculo"] for sub in allowed)

    filtered = df[df.apply(keep, axis=1)]
    if df.shape[0] > 0 and filtered.shape[0] == 0:
        logging.warning("⚠️ Todos os registros de '%s' foram removidos pelo filtro", df.name if hasattr(df, "name") else "")
    return filtered

# ─── 4) Aplicar filtro e registrar resultados ───────────────────────────────
filtered_dfs = {}
for tab, df in dfs.items():  # `dfs` vem da Célula 2
    filt = filter_df(df, campaigns_filter_lc, vehicles_map_lc)
    filtered_dfs[tab] = filt
    logging.info("🔍 '%s' após filtro: %d linhas (de %d)", tab, filt.shape[0], df.shape[0])

# ─── 5) Exibição rápida para inspeção ───────────────────────────────────────
for tab, df in filtered_dfs.items():
    print(f"\n📑 Prévia '{tab}' filtrado – top 3 linhas")
    display(df.head(3))


INFO | 2025-06-17 16:05:17 | 🔍 'modeloGeral' após filtro: 73 linhas (de 1887)
INFO | 2025-06-17 16:05:17 | 🔍 'modeloGenero' após filtro: 153 linhas (de 2361)
INFO | 2025-06-17 16:05:17 | 🔍 'modeloRegiao' após filtro: 1580 linhas (de 34251)
INFO | 2025-06-17 16:05:17 | 🔍 'modeloAlcance' após filtro: 57 linhas (de 1844)
INFO | 2025-06-17 16:05:17 | 🔍 'modeloIdade' após filtro: 263 linhas (de 8402)



📑 Prévia 'modeloGeral' filtrado – top 3 linhas


,date,account_name,campanha,id_campanha,veiculo,id_veiculo,ad_group_name,ad_name,start,end,...,video_play,video_watched_25,video_watched_50,video_watched_75,video_watched_100,post_reactions,post_shares,post_comments,engajamento_total,id
587,2025-06-06,Sebrae Nacional - DEBRITO,psmn,sbrae_2025_psmn,instagram,2,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,2025-06-06,2025-06-15,...,1884,1627,562,218,31,26,4,1,31,"2025-06-06-PSMN-82306-458,6-656"
588,2025-06-06,Sebrae Nacional - DEBRITO,psmn,sbrae_2025_psmn,instagram,2,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,2025-06-06,2025-06-15,...,8245,6479,3195,1301,795,255,13,7,275,"2025-06-06-PSMN-308679-2717,94-3331"
589,2025-06-06,Sebrae Nacional - DEBRITO,psmn,sbrae_2025_psmn,instagram,2,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_STORIES_GABRIELA BAILAS STORIES_ACAO...,2025-06-06,2025-06-15,...,137,733,302,181,129,2,0,0,2,"2025-06-06-PSMN-4689-51,03-82"



📑 Prévia 'modeloGenero' filtrado – top 3 linhas


,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,gender,impressions,cost,link_clicks,video_watched_100,id
862,2025-06-06,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,Tráfego,Mulher,74834,"403,44",578,29,"2025-06-06-PSMN-74834-403,44-578"
863,2025-06-06,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,Tráfego,Mulher,263720,"2298,66",2752,700,"2025-06-06-PSMN-263720-2298,66-2752"
864,2025-06-06,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_STORIES_GABRIELA BAILAS STORIES_ACAO...,Tráfego,Mulher,3900,"44,9",62,114,"2025-06-06-PSMN-3900-44,9-62"


...

,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,region,impressions,cost,link_clicks,video_watched_100,id
0,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_CARD_KV_ACAO_SBRAE_2025_PSMN0293,Tráfego,Acre,532,"1,71",4,0,"2025-06-15-PSMN-532-1,71-4"
1,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,Tráfego,Acre,26,"0,18",1,0,"2025-06-15-PSMN-26-0,18-1"
2,2025-06-15,Sebrae Nacional - DEBRITO,1,facebook,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,Tráfego,Acre,172,"1,04",6,0,"2025-06-15-PSMN-172-1,04-6"



📑 Prévia 'modeloAlcance' filtrado – top 3 linhas


,date,account_name,veiculo,id_veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,reach,impressions,id
587,2025-06-06,Sebrae Nacional - DEBRITO,facebook,1,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,Tráfego,71315,82306,2025-06-06-PSMN-82306--
588,2025-06-06,Sebrae Nacional - DEBRITO,facebook,1,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,Tráfego,274872,308679,2025-06-06-PSMN-308679--
589,2025-06-06,Sebrae Nacional - DEBRITO,facebook,1,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_STORIES_GABRIELA BAILAS STORIES_ACAO...,Tráfego,4227,4689,2025-06-06-PSMN-4689--



📑 Prévia 'modeloIdade' filtrado – top 3 linhas


,date,account_name,id_veiculo,veiculo,id_campanha,campanha,ad_group_name,ad_name,objective,age,impressions,cost,link_clicks,video_watched_100,id
2139,2025-06-06,Sebrae Nacional - DEBRITO,2,instagram,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_GABRIELA BAILAS REELS_ACAO_SBR...,Tráfego,25-34,56787,"242,83",325,15,"2025-06-06-PSMN-56787-242,83-325"
2140,2025-06-06,Sebrae Nacional - DEBRITO,2,instagram,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_REELS_VIC CERIDONO REELS_ACAO_SBRAE_...,Tráfego,25-34,126204,"875,03",848,220,"2025-06-06-PSMN-126204-875,03-848"
2141,2025-06-06,Sebrae Nacional - DEBRITO,2,instagram,sbrae_2025_psmn,psmn,2025_6_BR_TRAF_CPC_MULHERES DE 25 A 45 ANOS; I...,2025_6_BR_STORIES_GABRIELA BAILAS STORIES_ACAO...,Tráfego,25-34,1261,"12,24",8,26,"2025-06-06-PSMN-1261-12,24-8"


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 5 – PROJEÇÕES: verba, CPC projetado e volume_esperado            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# – Usa `filtered_dfs` da Célula 4
# – Detecta dinamicamente coluna de veículo (veiculo/vehicle/…)
# – Acrescenta: verba, custo_por_resultado_projetado, volume_esperado
# – Cria dict `projected_dfs`

from typing import Dict
import pandas as pd
import logging
from IPython.display import display

# 1) Parâmetros de planejamento (chaves em lower-case)
# Incluímos também "meta" para consolidar Facebook+Instagram
VERBA_BY_VEHICLE: Dict[str, float] = {
    "facebook":  50440,
    "instagram": 50440,
    "meta":      100880,   # soma de FB + IG
    "tiktok":    19369,
}
CPC_BY_VEHICLE: Dict[str, float] = {
    "facebook":  2.50,
    "instagram": 2.50,
    "meta":      2.50,     # CPC igual a FB/IG
    "tiktok":    4.80,
}

# 2) Possíveis nomes de coluna de veículo
POSSIBLE_VEHICLE_COLS = {"veiculo", "veículo", "vehicle", "placement"}

def detect_vehicle_col(df: pd.DataFrame) -> str | None:
    """
    Retorna o nome da coluna de veículo já normalizada para 'veiculo',
    renomeando quaisquer aliases em POSSIBLE_VEHICLE_COLS.
    """
    for col in df.columns:
        if col.lower() in POSSIBLE_VEHICLE_COLS:
            if col != "veiculo":
                df.rename(columns={col: "veiculo"}, inplace=True)
            return "veiculo"
    return None

# 3) Função que aplica as projeções
def add_projection_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # a) Garantir coluna 'veiculo'
    veh_col = detect_vehicle_col(df)
    if veh_col is None:
        logging.warning("⚠️ DataFrame sem coluna de veículo; pulando projeção.")
        return df

    # b) Normalizar valores de veiculo
    df["veiculo"] = (
        df["veiculo"]
          .astype(str)
          .str.strip()
          .str.casefold()
          .str.replace(r"\s+", " ", regex=True)
    )

    # c) Lookups de verba e CPC
    df["verba"] = df["veiculo"].map(VERBA_BY_VEHICLE)
    df["custo_por_resultado_projetado"] = df["veiculo"].map(CPC_BY_VEHICLE)

    # d) Avisar para veículos não mapeados
    for col, label in [("verba", "verba"), ("custo_por_resultado_projetado", "CPC")]:
        missing = df.loc[df[col].isna(), "veiculo"].unique()
        if len(missing):
            logging.warning(
                "⚠️ Veículo(s) sem %s definida: %s",
                label, ", ".join(map(str, missing))
            )

    # e) volume_esperado = round(verba / CPC) como Int64
    df["volume_esperado"] = (
        df["verba"] / df["custo_por_resultado_projetado"]
    ).round(0).astype("Int64")

    return df

# 4) Aplica projeções a cada modelo filtrado
projected_dfs: Dict[str, pd.DataFrame] = {}
for tab, df in filtered_dfs.items():          # `filtered_dfs` da Célula 4
    proj = add_projection_columns(df)
    projected_dfs[tab] = proj
    logging.info("🧮 '%s' → projeções aplicadas (%d linhas)", tab, proj.shape[0])

# 5) Visualização das primeiras linhas
for tab, df in projected_dfs.items():
    cols = [c for c in [
        "campanha", "veiculo", "impressions",
        "verba", "custo_por_resultado_projetado", "volume_esperado"
    ] if c in df.columns]
    print(f"\n📑 '{tab}' pós-projeção – top 3 linhas")
    display(df.head(3)[cols])


INFO | 2025-06-17 16:05:17 | 🧮 'modeloGeral' → projeções aplicadas (73 linhas)
INFO | 2025-06-17 16:05:17 | 🧮 'modeloGenero' → projeções aplicadas (153 linhas)
INFO | 2025-06-17 16:05:17 | 🧮 'modeloRegiao' → projeções aplicadas (1580 linhas)
INFO | 2025-06-17 16:05:17 | 🧮 'modeloAlcance' → projeções aplicadas (57 linhas)
INFO | 2025-06-17 16:05:17 | 🧮 'modeloIdade' → projeções aplicadas (263 linhas)



📑 'modeloGeral' pós-projeção – top 3 linhas


,campanha,veiculo,impressions,verba,custo_por_resultado_projetado,volume_esperado
587,psmn,instagram,82306,50440,2.5,20176
588,psmn,instagram,308679,50440,2.5,20176
589,psmn,instagram,4689,50440,2.5,20176



📑 'modeloGenero' pós-projeção – top 3 linhas


,campanha,veiculo,impressions,verba,custo_por_resultado_projetado,volume_esperado
862,psmn,facebook,74834,50440,2.5,20176
863,psmn,facebook,263720,50440,2.5,20176
864,psmn,facebook,3900,50440,2.5,20176


...

,campanha,veiculo,impressions,verba,custo_por_resultado_projetado,volume_esperado
0,psmn,facebook,532,50440,2.5,20176
1,psmn,facebook,26,50440,2.5,20176
2,psmn,facebook,172,50440,2.5,20176



📑 'modeloAlcance' pós-projeção – top 3 linhas


,campanha,veiculo,impressions,verba,custo_por_resultado_projetado,volume_esperado
587,psmn,facebook,82306,50440,2.5,20176
588,psmn,facebook,308679,50440,2.5,20176
589,psmn,facebook,4689,50440,2.5,20176



📑 'modeloIdade' pós-projeção – top 3 linhas


,campanha,veiculo,impressions,verba,custo_por_resultado_projetado,volume_esperado
2139,psmn,instagram,56787,50440,2.5,20176
2140,psmn,instagram,126204,50440,2.5,20176
2141,psmn,instagram,1261,50440,2.5,20176


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CÉLULA 6 – EXPORTAÇÃO + AUDITORIA DE IMPRESSÕES POR VEÍCULO             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# 1) Salva cada DataFrame de `projected_dfs` em CSV se contiver linhas;
#    emite WARNING se estiver vazio.
# 2) Reabre cada CSV salvo e compara a soma de `impressions` (Facebook / Instagram / TikTok)
#    com o DataFrame em memória.
# 3) Exibe tabela-resumo (df_sum × csv_sum) e faz WARNING se divergir.

from pathlib import Path
import pandas as pd
import logging
from pandas.errors import EmptyDataError
from IPython.display import display

# --- Configurações de veículos para auditoria -------------------------------
VEHICLES = ["facebook", "instagram", "tiktok"]
POSSIBLE_VEHICLE_COLS = ["veiculo", "veículo", "vehicle", "placement", "canal"]

def _ensure_vehicle_col(df: pd.DataFrame) -> pd.DataFrame:
    """Renomeia qualquer alias reconhecido para 'veiculo'."""
    for alt in POSSIBLE_VEHICLE_COLS:
        if alt in df.columns and alt != "veiculo":
            return df.rename(columns={alt: "veiculo"})
    return df

def impressions_sum(df: pd.DataFrame, vehicle: str) -> int:
    """Soma 'impressions' para um veículo (substring case-insensitive)."""
    if "impressions" not in df.columns:
        return 0
    df = _ensure_vehicle_col(df)
    if "veiculo" not in df.columns:
        return 0
    mask = df["veiculo"].str.contains(vehicle, case=False, na=False, regex=False)
    return int(df.loc[mask, "impressions"].sum())

# --- 1) Exportação dos CSVs ------------------------------------------------
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

saved_files: dict[str, Path] = {}
for tab, df in projected_dfs.items():  # `projected_dfs` da Célula 5
    if df.empty:
        logging.warning("⚠️ '%s' está vazio; CSV não gerado.", tab)
        continue
    csv_path = output_dir / f"{tab}.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    saved_files[tab] = csv_path
    logging.info(
        "💾 CSV salvo: %s (%d linhas, %d colunas)",
        csv_path,
        df.shape[0],
        df.shape[1],
    )

# --- 2) Auditoria df × CSV -------------------------------------------------
rows: list[dict[str, str | int]] = []
for vehicle in VEHICLES:
    row = {"Veículo": vehicle.title()}
    for tab, df in projected_dfs.items():
        df_sum = impressions_sum(df, vehicle)

        if tab not in saved_files:
            # CSV não gerado
            row[f"{tab}_df"] = df_sum
            row[f"{tab}_csv"] = "-"
            continue

        try:
            csv_df = pd.read_csv(saved_files[tab])
        except (EmptyDataError, pd.errors.ParserError):
            logging.warning("⚠️ CSV de '%s' está vazio ou inválido.", tab)
            csv_sum = 0
        else:
            csv_df = _ensure_vehicle_col(csv_df)
            csv_sum = impressions_sum(csv_df, vehicle)
            if df_sum != csv_sum:
                logging.warning(
                    "⚠️ Divergência em '%s' para %s: df=%d, csv=%d",
                    tab,
                    vehicle.title(),
                    df_sum,
                    csv_sum,
                )

        row[f"{tab}_df"] = df_sum
        row[f"{tab}_csv"] = csv_sum
    rows.append(row)

# --- 3) Exibição da auditoria ---------------------------------------------
audit_df = pd.DataFrame(rows).set_index("Veículo")
print("\n📊 Auditoria – Somatório de 'impressions' por veículo (DataFrame × CSV)")
display(audit_df)


INFO | 2025-06-17 16:05:17 | 💾 CSV salvo: output/modeloGeral.csv (73 linhas, 29 colunas)
INFO | 2025-06-17 16:05:17 | 💾 CSV salvo: output/modeloGenero.csv (153 linhas, 18 colunas)
INFO | 2025-06-17 16:05:17 | 💾 CSV salvo: output/modeloRegiao.csv (1580 linhas, 18 colunas)
INFO | 2025-06-17 16:05:17 | 💾 CSV salvo: output/modeloAlcance.csv (57 linhas, 15 colunas)
INFO | 2025-06-17 16:05:17 | 💾 CSV salvo: output/modeloIdade.csv (263 linhas, 18 colunas)



📊 Auditoria – Somatório de 'impressions' por veículo (DataFrame × CSV)


,modeloGeral_df,modeloGeral_csv,modeloGenero_df,modeloGenero_csv,modeloRegiao_df,modeloRegiao_csv,modeloAlcance_df,modeloAlcance_csv,modeloIdade_df,modeloIdade_csv
Veículo,,,,,,,,,,
Facebook,0,0,6808546,6808546,6866876,6866876,6866876,6866876,0,0
Instagram,6866876,6866876,0,0,0,0,0,0,6707027,6707027
Tiktok,6182257,6182257,6178150,6178150,6178150,6178150,6182257,6182257,6178150,6178150
